# Peformance comparison of different sparse data formats 

In this notebook we analyze the performance of various algorithms, including MLPs, dense dynamical systems, and various sorts of sparse dynamical systems.

In [20]:
import torch
import warnings
import time
warnings.filterwarnings("ignore")
from iterativennsimple.MaskedLinear import MaskedLinear
from iterativennsimple.MonarchLinear import MonarchLinear

import pandas as pd
import plotly.express as px
import numpy as np

import gc

In [21]:
results = {}

In [22]:
class moduleWrapper(object):
    def __init__(self, module, device):
        self.module = module
        self.device = device
    def __matmul__(self, x):
        return self.module(x.T).T

def generate(size, entries, matrix_types=["monarch","coo","csc","csr","dense","maskedLinear"], device="cuda"):
    """create a variety of sparse matrices 

    Args:
        size (int, optional): Size of the square matrix. Defaults to 1000.
        
        entries (int, optional): Total number of non-zero entries. Note this may not be exact, but should be close to the actual size. Defaults to 23*1000.
        
        matrix_types (list, optional): List of matrix types to generate. Defaults to ["monarch","coo","dense"].  Possible values are "monarch", "coo", "csc", "csr", "dense" and "maskedLinear".
        Must at least contain "coo", since that is used as the base for generating the other formats. 
        device (str, optional): "cuda" or "cpu". Defaults to "cuda".

    Returns:
        dict: Dictionary of the sparse matrices
    """ 
    if matrix_types is None:
        matrix_types = ["monarch","coo","csc","csr","dense","maskedLinear"]

    assert device in ["cuda", "cpu"], "device must be either 'cuda' or 'cpu'"
    assert "coo" in matrix_types, "matrix_types must contain 'coo' since that is used as the base for generating the other formats"

    output = {}
    if "monarch" in matrix_types:
        monarch_module = MonarchLinear.from_entry_target(size, size, entries, device=device)
        monarch = moduleWrapper(monarch_module, device)
        output["monarch"] = monarch
        if "coo" in matrix_types:
            coo = monarch_module.to_sparse_coo()
            output["coo"] = coo
    else:
        # We first create a COO tensor since that is easier to create
        indices = torch.randint(0, size, (entries,2))
        vals = torch.randn(entries)
        coo = torch.sparse_coo_tensor(indices.t(), vals, (size, size), device=device)
        coo = coo.coalesce()
        output["coo"] = coo

    # Then we convert it to CSC, CSR and dense
    if "dense" in matrix_types:
        dense = coo.to_dense()
        output["dense"] = dense
    if "csc" in matrix_types:
        csc = coo.to_sparse_csc()
        output["csc"] = csc
    if "csr" in matrix_types:
        csr = coo.to_sparse_csr()
        output["csr"] = csr
    if "maskedLinear" in matrix_types:         
        maskedLinear = moduleWrapper(MaskedLinear.from_coo(coo).to(device), device)
        output["maskedLinear"] = maskedLinear

    return output

In [23]:
## check that all of the matrices are the same operator
def matrix_check(size, entries, device):
    # Generate the matrices
    matrices = generate(size, entries, device=device)
    # Size of the RHS
    x_cols = 100
    x = torch.randn(size, x_cols, device=device)

    print('Matrix-matrix multiplication check')
    y_true = None
    base_name = None

    for matrix, matrix_name in zip(matrices.values(), matrices.keys()):
        y = matrix @ x
        if y_true is None:
            y_true = y
            base_name = matrix_name
        else:
            # print the frobenius norm of the difference
            print(f"||{matrix_name} - {base_name}||_F: {torch.norm(y-y_true)}")
            

In [24]:
matrices = generate(size=4, entries=8, device="cpu")
I = torch.eye(4, device="cpu")
for key in matrices.keys():
    print(key)
    M = matrices[key]@I
    print(M)

monarch
tensor([[-0.4600,  0.0000,  0.2541,  0.0000],
        [ 0.0000, -0.4646,  0.0000,  0.4199],
        [ 0.0000, -0.4755,  0.0000, -0.2996],
        [ 0.6448,  0.0000, -0.1086,  0.0000]], grad_fn=<PermuteBackward0>)
coo
tensor([[-0.4600,  0.0000,  0.2541,  0.0000],
        [ 0.0000, -0.4646,  0.0000,  0.4199],
        [ 0.0000, -0.4755,  0.0000, -0.2996],
        [ 0.6448,  0.0000, -0.1086,  0.0000]], grad_fn=<MmBackward0>)
dense
tensor([[-0.4600,  0.0000,  0.2541,  0.0000],
        [ 0.0000, -0.4646,  0.0000,  0.4199],
        [ 0.0000, -0.4755,  0.0000, -0.2996],
        [ 0.6448,  0.0000, -0.1086,  0.0000]], grad_fn=<MmBackward0>)
csc
tensor([[-0.4600,  0.0000,  0.2541,  0.0000],
        [ 0.0000, -0.4646,  0.0000,  0.4199],
        [ 0.0000, -0.4755,  0.0000, -0.2996],
        [ 0.6448,  0.0000, -0.1086,  0.0000]], grad_fn=<MmBackward0>)
csr
tensor([[-0.4600,  0.0000,  0.2541,  0.0000],
        [ 0.0000, -0.4646,  0.0000,  0.4199],
        [ 0.0000, -0.4755,  0.0000, -0.2996],

In [25]:
matrix_check(1000, 10*1000, "cpu")

Matrix-matrix multiplication check
||coo - monarch||_F: 1.487124518462224e-05
||dense - monarch||_F: 1.3757422493654303e-05
||csc - monarch||_F: 1.4400521649804432e-05
||csr - monarch||_F: 1.4400521649804432e-05
||maskedLinear - monarch||_F: 1.3716979083255865e-05


In [26]:
matrix_check(1000, 10*1000, "cuda")

Matrix-matrix multiplication check
||coo - monarch||_F: 1.5399298717966303e-05
||dense - monarch||_F: 1.465857531002257e-05
||csc - monarch||_F: 1.5399298717966303e-05
||csr - monarch||_F: 1.5399298717966303e-05
||maskedLinear - monarch||_F: 1.4986351743573323e-05


In [27]:
def run_timing(size, entries, device, syncgpu, matrix_types=None):
    # test if cuda is available
    if device == "cuda":
        if not torch.cuda.is_available():
            print("CUDA is not available, using CPU instead")
            device = "cpu"

    print(f"Running on {device}")
    print(f"Size: {size}")
    print(f"Entries: {entries}")
    print(f"Synchronize GPU: {syncgpu}")
    
    # Generate the matrices
    matrices = generate(size, entries, matrix_types=matrix_types, device=device)

    # Size of the RHS
    x_cols = 100

    # Compute the timings
    print('Matrix-matrix multiplication timings:')
    base_time = None
    base_name = None
    all_times = {}
    for matrix, matrix_name in zip(matrices.values(), matrices.keys()):
        # first, do a few runs to warm up the cache
        for i in range(2):
            x = torch.randn(size, x_cols, device=device)
            y = matrix @ x
        # now do the timings
        matrix_times = []
        for i in range(5):
            x = torch.randn(size, x_cols, device=device)    
            if syncgpu:
                torch.cuda.synchronize()
            start = time.perf_counter()
            y = matrix @ x
            if syncgpu:
                torch.cuda.synchronize()
            matrix_time = time.perf_counter()-start
            matrix_times.append(matrix_time)
        avg_matrix_time = sum(matrix_times)/len(matrix_times)
        min_matrix_time = min(matrix_times)
        max_matrix_time = max(matrix_times)
        if base_time is None:
            base_time = avg_matrix_time
            base_name = matrix_name

        name = device+'_'+matrix_name
        all_times[name] = avg_matrix_time

        print(f"{name} avg time:", avg_matrix_time)
        print(f"{name} min time:", min_matrix_time)
        print(f"{name} max time:", max_matrix_time)
        print(f"{name} speedup over {base_name}:", base_time/avg_matrix_time)

    if torch.cuda.is_available():
        del matrices
        # Clear the GPU memory cache
        torch.cuda.empty_cache()
        gc.collect()

    return all_times

In [28]:
def make_plot(matrix_size, matrix_entries, device, matrix_types=None):
    print("matrix size:", matrix_size)
    print("Total number of entries:", matrix_size**2)
    print("Active entries:", int(matrix_entries))
    print("Ratio of active entries to total entries:", matrix_entries/(matrix_size**2))
    # Print the amount of free memory in GB
    if torch.cuda.is_available():
        free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
        print(f"Free memory on GPU: {free_memory / (1024**3):.2f} GB")

    results = run_timing(matrix_size, matrix_entries, matrix_types=matrix_types, 
                         device=device, syncgpu=True)

    # Create a dataframe with row and column names for the heatmap
    df = pd.DataFrame(columns=results.keys(), index=results.keys())

    # Fill in df with the values we want to display
    for name1, time1 in results.items():
        for name2, time2 in results.items():
            df.loc[name1, name2] = time1/time2
    # Create the heatmap and make it large
    # Plot the original heatmap
    fig = px.imshow(df, text_auto=True, color_continuous_scale='Jet', width=1000, height=900, title="Relative Timing Heatmap")
    fig.show()

    # Plot the log10 heatmap
    df_log = np.log10(df.astype(float))
    fig_log = px.imshow(df_log, text_auto=True, color_continuous_scale='Jet', width=1000, height=900, title="Log10 Relative Timing Heatmap")
    fig_log.show()

In [29]:
matrix_size = 1024*16
matrix_entries = (matrix_size**2)/2
device = "cuda"
matrix_types = ["monarch","coo","csc","csr","dense","maskedLinear"]
make_plot(matrix_size, matrix_entries, device, matrix_types=matrix_types)

matrix size: 16384
Total number of entries: 268435456
Active entries: 134217728
Ratio of active entries to total entries: 0.5
Free memory on GPU: 23.50 GB
Running on cuda
Size: 16384
Entries: 134217728.0
Synchronize GPU: True
Matrix-matrix multiplication timings:
cuda_monarch avg time: 0.0011773512000218035
cuda_monarch min time: 0.0011675389832817018
cuda_monarch max time: 0.0011984269949607551
cuda_monarch speedup over monarch: 1.0
cuda_coo avg time: 0.02634407519362867
cuda_coo min time: 0.0263267089612782
cuda_coo max time: 0.02635615400504321
cuda_coo speedup over monarch: 0.04469130881871103
cuda_dense avg time: 0.0014918012078851462
cuda_dense min time: 0.0014734140131622553
cuda_dense max time: 0.0015378560055978596
cuda_dense speedup over monarch: 0.7892145372980873
cuda_csc avg time: 0.10185132460901514
cuda_csc min time: 0.10159135702997446
cuda_csc max time: 0.1023059930303134
cuda_csc speedup over monarch: 0.011559507984225007
cuda_csr avg time: 0.013924787216819823
cuda_c

In [30]:
matrix_size = 1024*16
matrix_entries = (matrix_size**2)/8
device = "cuda"
matrix_types = ["monarch","coo","csc","csr","dense","maskedLinear"]
make_plot(matrix_size, matrix_entries, device, matrix_types=matrix_types)

matrix size: 16384
Total number of entries: 268435456
Active entries: 33554432
Ratio of active entries to total entries: 0.125
Free memory on GPU: 23.50 GB
Running on cuda
Size: 16384
Entries: 33554432.0
Synchronize GPU: True
Matrix-matrix multiplication timings:
cuda_monarch avg time: 0.0010214435984380542
cuda_monarch min time: 0.0009563709609210491
cuda_monarch max time: 0.001183127984404564
cuda_monarch speedup over monarch: 1.0
cuda_coo avg time: 0.006632085400633514
cuda_coo min time: 0.006629655021242797
cuda_coo max time: 0.006636077014263719
cuda_coo speedup over monarch: 0.1540154471383139
cuda_dense avg time: 0.001427582185715437
cuda_dense min time: 0.0014176190015859902
cuda_dense max time: 0.0014438089565373957
cuda_dense speedup over monarch: 0.7155059853357268
cuda_csc avg time: 0.024465990206226706
cuda_csc min time: 0.02444993396056816
cuda_csc max time: 0.024489699047990143
cuda_csc speedup over monarch: 0.041749530259277724
cuda_csr avg time: 0.0029862837982364
cuda

In [31]:
def train_least_squares(size, entries, device, syncgpu, epochs=20):
    # test if cuda is available
    if device == "cuda":
        if not torch.cuda.is_available():
            print("CUDA is not available, using CPU instead")
            device = "cpu"

    print(f"Running on {device}")
    print(f"Size: {size}")
    print(f"Entries: {entries}")
    print(f"Synchronize GPU: {syncgpu}")
    
    # Generate the matrices
    matrices = generate(size, entries, device=device)

    # Generate the data
    X = torch.randn(10, size, device=device)  # Example input data
    Y = torch.randn(10, size, device=device)  # Example target data

    # Define the loss function
    criterion = torch.nn.MSELoss()

    # Wrap a model around the sparse matrix
    class matrixWrapper(torch.nn.Module):
        def __init__(self, matrix, type="dense"):
            super(matrixWrapper, self).__init__()
            self.type = type
            self.matrix = torch.nn.Parameter(matrix)
        def forward(self, x):
            if self.type == "dense":
                return torch.mm(x, self.matrix.T)
            else:
                return torch.sparse.mm(self.matrix, x.T).T

    # Train the models
    print('Training least squares models:')
    for matrix, matrix_name in zip(matrices.values(), matrices.keys()):
        if matrix_name in ["coo", "csr"]:
            sparse_model = matrixWrapper(matrix, type="sparse").to(device)
        elif matrix_name in ["dense"]:
            sparse_model = matrixWrapper(matrix, type="dense").to(device)
        elif matrix_name in ["maskedLinear"]:
            sparse_model = matrix.module.to(device)
        elif matrix_name in ["monarch"]:
            sparse_model = matrix.module.to(device)
        else:
            continue



        try:
            # Define the optimizer
            # optimizer = torch.optim.SGD(sparse_model.parameters(), lr=0.01)
            # print('optimize: SGD')
            optimizer = torch.optim.Adam(sparse_model.parameters(), lr=0.01)
            print('optimize: Adam')
            # Training loop
            print(f'-------------{matrix_name} training:--------------')
            for epoch in range(epochs):
                optimizer.zero_grad()
                output = sparse_model(X)
                loss = criterion(output, Y)
                if epoch % 10 == 0:
                    print(f'{matrix_name} Epoch {epoch}, Loss: {loss.item()}')
                loss.backward()
                optimizer.step()

            print(f'{matrix_name} final loss: {loss.item()}')
        except Exception as e:
            print(f"Error training {matrix_name}: {e}")
            continue
    return matrices

# Example usage
matrices = train_least_squares(size=100, entries=500, device='cuda', syncgpu=True)

Running on cuda
Size: 100
Entries: 500
Synchronize GPU: True
Training least squares models:
optimize: Adam
-------------monarch training:--------------
monarch Epoch 0, Loss: 1.3640310764312744
monarch Epoch 10, Loss: 1.0733706951141357
monarch final loss: 0.9036344289779663
optimize: Adam
-------------coo training:--------------
coo Epoch 0, Loss: 1.3640309572219849
Error training coo: Adam does not support sparse gradients, please consider SparseAdam instead
optimize: Adam
-------------dense training:--------------
dense Epoch 0, Loss: 1.3640310764312744
dense Epoch 10, Loss: 0.14665405452251434
dense final loss: 0.04092293605208397
optimize: Adam
-------------csr training:--------------
csr Epoch 0, Loss: 1.3640309572219849
Error training csr: unsupported tensor layout: SparseCsr
optimize: Adam
-------------maskedLinear training:--------------
maskedLinear Epoch 0, Loss: 1.3640310764312744
maskedLinear Epoch 10, Loss: 1.0733706951141357
maskedLinear final loss: 0.9036344289779663
